# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [1]:
import os

# Async CUDA allocator
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# If cuDNN autotune fails, fall back to a safe (but slower) algorithm.
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true' 

### 1.2. Imports

In [2]:
from _imports import * # Centralized file containing all imports

2025-05-24 11:36:23.267821: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-24 11:36:23.277927: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748097383.289763   18309 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748097383.293166   18309 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-24 11:36:23.305211: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

### 1.3. GPU Management

In [3]:
# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
get_gpu_info()

TensorFlow Version: 2.18.0
CUDA support detected
  CUDA Version: 12.5.1
  cuDNN Version: 9

GPUs Detected (1): ['/physical_device:GPU:0']
Default GPU device: /device:GPU:0


2025-05-24 11:36:24.526920: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1748097384.526991   18309 gpu_process_state.cc:201] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1748097384.527156   18309 gpu_device.cc:2022] Created device /device:GPU:0 with 2278 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


## 2. Run Parameters 

In [4]:
NUM_TRIALS = 10
EPOCHS = 3

SEED = 42

mixed_precision.set_global_policy("mixed_float16")

In [5]:
TOP_K = 1  # Number of top trials to save

# True -> the greatest, the better
# False -> the least, the better
RANK_DESCENDING = False  

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "value" 

In [6]:
# Set to an existing path to resume training
RESUME_TRAINING_PATH = "runs/nas_cnn3d_v0" # None or "runs/nas_1" 

RUN_DIR = RESUME_TRAINING_PATH or create_run_directory(prefix="nas_")

## 3. Data Loading and Preprocessing

In [7]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels()

/media/matheus/SSD/Projects/RayWise/src/_load_dataset.py:42: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_train = s008_y_train.astype(np.float32)
/media/matheus/SSD/Projects/RayWise/src/_load_dataset.py:65: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_val = s008_y_val.astype(np.float32)


Shape before conversion: (9234, 8, 32)
Shape after conversion: (9234,)
y_train shape: (9234,)
coord_input shape: (9234, 2)
lidar_input shape: (9234, 20, 200, 10)
Shape before conversion: (1960, 8, 32)
Shape after conversion: (1960,)
y_val shape: (1960,)
coord_input_val shape: (1960, 2)
lidar_input_val shape: (1960, 20, 200, 10)
y_train shape: (11194,)
coord_input shape: (11194, 2)
lidar_input shape: (11194, 20, 200, 10)
Shape before conversion: (9638, 8, 32)
Shape after conversion: (9638,)
y shape: (9638,)
coord_input shape: (9638, 2)
lidar_input shape: (9638, 20, 200, 10)


/media/matheus/SSD/Projects/RayWise/src/_load_dataset.py:101: ComplexWarning: Casting complex values to real discards the imaginary part
  s009_y = s009_y.astype(np.float32)


In [8]:
(
    x_s008_lidar_train,
    x_s008_lidar_val,
    x_s008_coord_train,
    x_s008_coord_val,
    y_s008_train,
    y_val,
) = train_test_split(
    s008_lidar_input,
    s008_coord_input,
    s008_y_train,
    test_size=0.2,
    random_state=SEED,
    shuffle=True,
)

(
    x_s009_lidar_test,
    x_s009_lidar_val,
    x_s009_coord_test,
    x_s009_coord_val,
    y_s009_test,
    y_s009_val,
) = train_test_split(
    s009_lidar_input,
    s009_coord_input,
    s009_y,
    test_size=0.2,
    random_state=SEED,
    shuffle=True,
)

x_lidar_train = x_s008_lidar_train
x_coord_train = x_s008_coord_train
y_train = y_s008_train

x_lidar_val = np.concatenate((x_s008_lidar_val, x_s009_lidar_val), axis=0)
x_coord_val = np.concatenate((x_s008_coord_val, x_s009_coord_val), axis=0)
y_val = np.concatenate((y_val, y_s009_val), axis=0)

x_lidar_test = x_s009_lidar_test
x_coord_test = x_s009_coord_test
y_test = y_s009_test

## 4. Getters

### 4.1. Callbacks

In [9]:
def get_callbacks(trial: optuna.Trial, backup_dir: str) -> List[tf.keras.callbacks.Callback]:
    """
    Constructs and returns a list of Keras callbacks tailored for Optuna trials.

    Args:
        trial (optuna.Trial): The current Optuna trial object.
        backup_dir (str): Directory where the backup files will be stored.

    Returns:
        List[tf.keras.callbacks.Callback]: A list of callbacks to pass into `model.fit()`.
    """
    # Metric to monitor for early stopping and checkpointing
    monitor: str = "val_loss"

    # Stop training early if no improvement in validation loss for N epochs
    early_stopping = callbacks.EarlyStopping(
        monitor=monitor,
        patience=6,  # number of epochs to wait
        restore_best_weights=True,
        verbose=1,
    )

    # Reduce learning rate if validation loss plateaus
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor=monitor,
        patience=3,  # how many epochs to wait before reducing LR
        factor=0.2,  # reduce LR by this factor
        min_lr=1e-6,  # don't reduce below this
        verbose=1,
    )
    
    # Backup and restore the model
    # backup = callbacks.BackupAndRestore(backup_dir=backup_dir)
    
    # Model checkpointing
    # checkpoint = callbacks.ModelCheckpoint(
    #     filepath=os.path.join(backup_dir, "checkpoint.h5"),
    #     monitor=monitor,
    #     save_best_only=True,
    #     save_weights_only=True,
    # )

    #! ——————— WARNING: the callbacks below do not work with multi-objective —————— !#
    # Custom callback to prune trial if NaN loss is encountered
    nan_pruner_callback = callbacks.TerminateOnNaN()

    # Optuna's built-in pruning callback for early trial termination
    pruning_callback = KerasPruningCallback(trial, monitor, interval=3)
    #! ———————————————————————————————————————————————————————————————————————————— !#

    # Return the complete list of callbacks
    return [early_stopping, reduce_lr, nan_pruner_callback, pruning_callback]

## 5. Hyperparameters

In [10]:
hparams = HParams(
    activation_choices=[
        "relu",
        "tanh",
        "sigmoid",  # Logistic
        "swish",  # x * sigmoid(x)
    ],
    regularizer_choices=[
        "none",
        "l1",
        "l2",
        "l1l2",
    ],
    optimizer_choices=[
        "AdamW",
        "Lion",
        "RMSprop",
        # "Adam",
        # "Nadam",
        # "SGD",
    ],
    scaler_choices=[
        "StandardScaler",
        "MinMaxScaler_0_1",
        "MinMaxScaler_-1_1",
        # "RobustScaler",
        # "QuantileTransformer",
        # "PowerTransformer",
    ],
    l1_value=1e-2,
    l2_value=1e-2,
    min_lr=1e-5,
    max_lr=1e-2,
)

initializer_options = [
    initializers.Zeros(),
    initializers.Ones(),
    initializers.Constant(),
    initializers.RandomNormal(),
    initializers.RandomUniform(),
    initializers.TruncatedNormal(),
    initializers.GlorotNormal(),
    initializers.GlorotUniform(),
    initializers.HeNormal(),
    initializers.HeUniform(),
    initializers.LecunNormal(),
    initializers.LecunUniform(),
    initializers.Identity(),
    initializers.Orthogonal(),
    initializers.VarianceScaling(),
]


## 6. Objective Function

In [11]:
def objective(
    trial: optuna.Trial,
    backup_dir: str,
    model_dir: str,
    fig_dir: str,
    logs_dir: str,
    history_dir: str,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
    show_summary: bool = False,
    plot_model: bool = False,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        X (List[np.ndarray]): List of input arrays.
        y (List[np.ndarray]): List of label arrays.
        backup_dir (str): Path to store backup files.
        model_dir (str): Path to store full models.
        fig_dir (str): Path to store plots.
        logs_dir (str): Path to store logs.
        history_dir (str): Path to store training history.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.
        show_summary (bool): If True, display the model summary.
        plot_model (bool): If True, display a plot of the model architecture.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """

    global x_lidar_train
    global x_coord_train
    global y_train
    global x_lidar_val
    global x_coord_val
    global y_val
    global x_lidar_test
    global x_coord_test
    global y_test
    global s009_lidar_input
    global s009_coord_input
    global s009_y

    # Each trial gets a different seed
    np.random.seed(trial.number)
    tf.random.set_seed(trial.number)

    # ———————————————————————————————————————————————————————————————————————————— #

    model = None
    try:

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Model Construction                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ———————————————————— Decide on how to normalize the data ——————————————————— #
        # ? These were the best performing options in the previous trials
        lidar_norm_type: str = (
            "none"  # trial.suggest_categorical("lidar_norm", ["none", "minmax", "standard"])
        )
        coord_norm_type: str = "standard"  # trial.suggest_categorical("coord_norm", ["minmax", "standard"])

        if lidar_norm_type == "minmax":
            lidar_scaler = MinMaxScaler(feature_range=(-1, 1))
        elif lidar_norm_type == "standard":
            lidar_scaler = StandardScaler()
        else:
            lidar_scaler = None

        if coord_norm_type == "minmax":
            coord_scaler = MinMaxScaler(feature_range=(-1, 1))
        elif coord_norm_type == "standard":
            coord_scaler = StandardScaler()

        coord_scaler.fit(x_coord_train)  # Not cheating and using val data
        x_coord_train = coord_scaler.transform(x_coord_train)
        x_coord_val = coord_scaler.transform(x_coord_val)
        x_coord_test = coord_scaler.transform(x_coord_test)
        s009_coord_input = coord_scaler.transform(s009_coord_input)

        if lidar_scaler is not None:
            flat_train = x_lidar_train.reshape(-1, x_lidar_train.shape[-1])
            lidar_scaler.fit(flat_train)  # Not cheating and using val data

            def _scale_lidar(arr: np.ndarray) -> np.ndarray:
                flat = arr.reshape(-1, arr.shape[-1])
                scaled = lidar_scaler.transform(flat)
                return scaled.reshape(arr.shape)

            x_lidar_train = _scale_lidar(x_lidar_train)
            x_lidar_val = _scale_lidar(x_lidar_val)
            x_lidar_test = _scale_lidar(x_lidar_test)
            s009_lidar_input = _scale_lidar(s009_lidar_input)

        # ——————————————————————————————— LiDAR Input ——————————————————————————————— #
        # Input for LiDAR data (e.g., shape: (20, 200, 10))
        x_lidar_input = layers.Input(shape=(20, 200, 10))

        # Reshape to (batch, 20, 200, 10, 1) #! Channel last
        x_lidar_input = layers.Reshape((20, 200, 10, 1))(x_lidar_input)

        # ———————————————————————————————— GPS Input ———————————————————————————————— #
        # Input for coordinate data (e.g., shape: (2,))
        x_coord_input = layers.Input(shape=(x_coord_train.shape[1],))

        # 1×1×1 spatial, 2 channels
        x_coord = layers.Reshape((1, 1, 1, x_coord_input.shape[-1]))(x_coord_input)

        # tile to 20×200×10 spatial dims -> (batch, 20, 200, 10, 2)
        x_coord = layers.Lambda(
            lambda x: tf.tile(x, [1, 20, 200, 10, 1]),
            #! Lambda has deserialization issues, so providing the output shape is necessary
            output_shape=(20, 200, 10, 2),
        )(x_coord)

        # ————————————————————————————— Combine Branches ————————————————————————————— #
        # Fuse channels: (batch,20,200,10,1) + (batch,20,200,10,2) -> (batch,20,200,10,3)
        combined = layers.Concatenate(axis=-1)([x_lidar_input, x_coord])

        # ? These were the best performing options in the previous trials
        max_layers = 3  # trial.suggest_int("num_layers", 1, 4)

        # Calculate max pool size
        max_pool_dim1 = math.floor(20 ** (1.0 / max_layers))
        max_pool_dim2 = math.floor(200 ** (1.0 / max_layers))
        max_pool_dim3 = math.floor(10 ** (1.0 / max_layers))

        # ? Tip: after finding max_layers, for each layer call a different builder, then optimize it individually
        x = model = build_cnn3d(
            trial=trial,
            hparams=hparams,
            x=combined,
            max_layers=max_layers,
            max_filters=512,
            min_filters=32,
            filters_step=32,
            max_kernel_size=5,
            min_kernel_size=1,
            min_pool_size_dim1=1,
            max_pool_size_dim1=max_pool_dim1,
            min_pool_size_dim2=1,
            max_pool_size_dim2=max_pool_dim2,
            min_pool_size_dim3=1,
            max_pool_size_dim3=max_pool_dim3,
            data_format="channels_last",
            padding="same",
            strides=(1, 1, 1),
            dilation_rate=(1, 1, 1),
            groups=1,
            # kernel_initializer=trial.suggest_categorical("initializer_layer_1", initializer_options),
            trial_batch_norm=True,
            trial_kernel_reg=False,
            trial_bias_reg=False,
            trial_activity_reg=False,
            regularizer_positions=None,
            trial_skip_connections=False,
            share_activation=False,
            name_prefix="cnn3d",
        )

        # ———————————————————————————— Flatten the Output ———————————————————————————— #
        x = layers.Flatten(name="flatten")(x)

        # ——————————————————————————————— Dense Layers ——————————————————————————————— #
        # ? This was the best performing option in the previous trials
        num_dense_layers = 2 # trial.suggest_int("num_dense_layers", 0, 3)
        for i in range(num_dense_layers):
            x = build_dnn(
                trial=trial,
                hparams=hparams,
                x=x,
                max_layers=1,
                max_units=500,
                min_units=50,
                units_step=50,
                # kernel_initializer=trial.suggest_categorical("initializer_layer_1", initializer_options),
                min_dropout_rate=0.0,
                max_dropout_rate=0.5,
                dropout_rate_step=0.1,
                dropout_positions=None,  # None for all layers
                regularizer_positions=None,  # None for all layers
                trial_batch_norm=False,
                trial_kernel_reg=False,
                trial_bias_reg=False,
                trial_activity_reg=False,
                trial_skip_connections=False,
                share_activation=False,
                name_prefix=f"extra_dense_{i}",
            )

        # —————————————————————————————————— Output —————————————————————————————————— #
        outputs = layers.Dense(256, activation="softmax")(x)

        # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
        model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

        # ———————————————————————————— Vizualize the Model ——————————————————————————— #
        if show_summary:
            model.summary()

        if plot_model:
            # Display the model architecture image
            tf.keras.utils.plot_model(
                model,
                to_file=os.path.join(fig_dir, f"model_plot_{trial.number}.png"),
                show_shapes=True,
                show_layer_names=True,
            )
            display(Image(filename=os.path.join(fig_dir, f"model_plot_{trial.number}.png")))

        # ————————————————————————————— Compile the Model ———————————————————————————— #
        optimizer = hparams.get_optimizer(trial)
        model.compile(
            optimizer=optimizer,
            loss=losses.SparseCategoricalCrossentropy(),
            metrics=["accuracy"],
        )

        # ———————————————————————————————— Train Model ——————————————————————————————— #
        batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
        history = model.fit(
            [x_lidar_train, x_coord_train],
            y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=get_callbacks(trial, backup_dir),
            verbose=2,
        )

        params = model.count_params()
        trial.set_user_attr("num_params", params)
        trial.set_user_attr("model_size_mb", params * 4 / (1024**2))

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss = min(history.history["val_loss"])
        if size_penalizer == "flops":
            loss = compute_flops_penalized_loss(loss=loss, model=model)
        elif size_penalizer == "params":
            loss = compute_params_penalized_loss(loss=loss, model=model)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                                 Trial Results                                #
        # ———————————————————————————————————————————————————————————————————————————— #
        clear_output(wait=True)

        epochs = list(range(1, len(history.history["loss"]) + 1))
        train_loss = history.history["loss"]
        val_loss = history.history["val_loss"]
        train_acc = history.history.get("accuracy", [])
        val_acc = history.history.get("val_accuracy", [])
        val_loss_best = min(history.history["val_loss"])

        # Create figure with two subplots
        fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(16, 6))

        # Left: Loss
        ax_loss.plot(epochs, train_loss, marker="o", linestyle="-", label="Training Loss")
        ax_loss.plot(epochs, val_loss, marker="x", linestyle="--", label="Validation Loss")
        ax_loss.set_title("Training & Validation Loss")
        ax_loss.set_xlabel("Epoch")
        ax_loss.set_ylabel("Loss")
        ax_loss.set_xticks(epochs)
        ax_loss.set_ylim(0, max(max(train_loss), max(val_loss)) * 1.05)
        ax_loss.grid(True)
        ax_loss.legend()

        # Right: Accuracy (if available)
        if train_acc and val_acc:
            ax_acc.plot(epochs, train_acc, marker="v", linestyle="-", label="Training Accuracy")
            ax_acc.plot(epochs, val_acc, marker="^", linestyle="--", label="Validation Accuracy")
            ax_acc.set_title("Training & Validation Accuracy")
            ax_acc.set_xlabel("Epoch")
            ax_acc.set_ylabel("Accuracy")
            ax_acc.set_xticks(epochs)
            ax_acc.set_ylim(0, 1)
            ax_acc.grid(True)
            ax_acc.legend()

            trial.set_user_attr("best_train_accuracy", float(max(train_acc)))
            trial.set_user_attr("best_val_accuracy", float(max(val_acc)))
        else:
            ax_acc.axis("off")  # hide if accuracy not present

        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, f"trial_{trial.number}.png"), dpi=300)
        plt.close(fig)

        # ————————————————————————————— Evaluate on s009 ————————————————————————————— #
        test_loss, test_acc = model.evaluate(
            [x_lidar_test, x_coord_test], y_test, batch_size=batch_size, verbose=0
        )

        trial.set_user_attr("test_accuracy_s009", float(test_acc))

        # Now evaluate on the full s009 dataset for comparison purposes
        test_loss_full, test_acc_full = model.evaluate(
            [s009_lidar_input, s009_coord_input], s009_y, batch_size=batch_size, verbose=0
        )
        trial.set_user_attr("test_accuracy_s009_full", float(test_acc_full))

        # ————————————————————————————— Print the results ———————————————————————————— #

        print(f"\n\n# ——————————————————————— Trial {trial.number} Results ——————————————————————— #")
        print("\n" + "=" * 15)
        print(f"Training loss: {loss:.12f}")
        print(f"Training accuracy: {max(train_acc):.4f}\n")
        print(f"Validation loss: {val_loss_best:.12f}")
        print(f"Validation accuracy: {max(val_acc):.4f}\n")
        print(f"Test loss (s009): {test_loss:.12f}")
        print(f"Test accuracy (s009): {test_acc:.4f}\n")
        print(f"Test loss (s009 full): {test_loss_full:.12f}")
        print(f"Test accuracy (s009 full): {test_acc_full:.4f}\n")

        params = model.count_params()
        print(f"Number of parameters: {params}")
        print(f"Model size: {params * 4 / (1024 ** 2):.2f} MB")
        print("=" * 15 + "\n")
        print("# ———————————————————————————————————————————————————————————————————————————— #\n\n")

        # ——————————————————————————————— Save history ——————————————————————————————— #
        # Save training history to CSV
        history_path = os.path.join(history_dir, f"trial_{trial.number}.csv")

        # Create a DataFrame with all history data
        history_data = {
            "epoch": list(range(1, len(history.history["loss"]) + 1)),
            "train_loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
        }

        # Add accuracy metrics if available
        if "accuracy" in history.history:
            history_data["train_accuracy"] = history.history["accuracy"]
        if "val_accuracy" in history.history:
            history_data["val_accuracy"] = history.history["val_accuracy"]

        # Convert to DataFrame and save as CSV
        history_df = pd.DataFrame(history_data)
        history_df.to_csv(history_path, index=False)

        return loss

    except optuna.exceptions.TrialPruned:
        raise  # simply propagate pruning
    except tf.errors.ResourceExhaustedError as oom_err:
        # Catch OOM / resource exhausted
        print(f"❌ Trial {trial.number} hit OOM (ResourceExhaustedError): {oom_err}")
        log_exception_to_file(title=f"oom_trial_{trial.number}", error=oom_err, logs_dir=logs_dir)

        return float("inf")  # Return bad loss
    except Exception as e:
        print(f"An error occurred during the trial execution: {e}")
        log_exception_to_file(title=f"error_trial_{trial.number}", error=e, logs_dir=logs_dir)

        raise  # Re-raise the exception to propagate it
    finally:
        if model is not None:
            clear_session()
            del model

## 7. Code Health Check

In [12]:
# resources_dir = os.path.join(RUN_DIR, "resources")
# os.makedirs(resources_dir, exist_ok=True)
# log_resources(log_dir=resources_dir)

In [13]:
_monitor_proc = start_monitor(
    pid=os.getpid(),
    log_dir=RUN_DIR,
    custom_title=f"{RUN_DIR}",
    recipients_file="./json/recipients.json",
    credentials_file="./json/credentials.json",
)

## Main

In [ ]:
try:
    # ——————————————————————————————— Storage paths —————————————————————————————— #
    study_dir = os.path.join(RUN_DIR, f"optuna_study")
    os.makedirs(study_dir, exist_ok=True)

    dirs = {
        "args": os.path.join(study_dir, "args"),
        "figures": os.path.join(study_dir, "figures"),
        "backup": os.path.join(study_dir, "backup"),
        "history": os.path.join(study_dir, "history"),
        "models": os.path.join(study_dir, "models"),
        "logs": os.path.join(study_dir, "logs"),
    }
    for path in dirs.values():
        os.makedirs(path, exist_ok=True)

    storage_path = f"sqlite:///{os.path.join(study_dir, 'optuna_study.db')}"
    backup_dir, history_dir, model_dir, fig_dir, args_dir, logs_dir = (
        dirs["backup"],
        dirs["history"],
        dirs["models"],
        dirs["figures"],
        dirs["args"],
        dirs["logs"],
    )

    # —————————————————————————————————— Pruners ————————————————————————————————— #
    pruner = optuna.pruners.HyperbandPruner()

    # ——————————————————————————————————— Study —————————————————————————————————— #
    study = optuna.create_study(
        study_name=os.path.basename(study_dir),
        storage=storage_path,
        direction="minimize",
        pruner=pruner,
        load_if_exists=True,
    )

    # Count trials done, then determine the remaining trials
    done_trials = len(
        study.get_trials(
            deepcopy=False,
            states=(
                optuna.trial.TrialState.COMPLETE,
                optuna.trial.TrialState.PRUNED,
                optuna.trial.TrialState.FAIL,
            ),
        )
    )
    n_remaining_trials = max(0, NUM_TRIALS - done_trials)

    study.optimize(
        lambda trial: objective(
            trial,
            backup_dir=backup_dir,
            model_dir=model_dir,
            fig_dir=fig_dir,
            logs_dir=logs_dir,
            history_dir=history_dir,
            epochs=EPOCHS,
            size_penalizer=None,
            show_summary=True,
        ),
        n_trials=n_remaining_trials,
        catch=(ValueError, RuntimeError),
        gc_after_trial=True,
        n_jobs=1,  # If you have multiple GPUs/Cores
        show_progress_bar=False,
    )

    # ————————————————————————————— Save Top-K Trials ———————————————————————————— #
    valid_trials = [
        t for t in study.trials if t.value is not None and not (math.isnan(t.value) or math.isinf(t.value))
    ]

    sorted_trials = sorted(
        valid_trials,
        key=lambda t: (t.value if RANK_KEY == "value" else t.user_attrs.get(RANK_KEY, float("nan"))),
        reverse=RANK_DESCENDING,  # True -> Descending order
    )[:TOP_K]

    for rank, trial in enumerate(sorted_trials):
        trial_id = trial.number
        trial_params = trial.params
        trial_loss = trial.value
        trial_num_params = trial.user_attrs.get("num_params", None)
        trial_model_size = trial.user_attrs.get("model_size_mb", None)

        trial_train_acc = trial.user_attrs.get("best_train_accuracy", None)
        trial_val_acc = trial.user_attrs.get("best_val_accuracy", None)
        trial_test_acc = trial.user_attrs.get("test_accuracy_s009", None)
        trial_test_acc_full = trial.user_attrs.get("test_accuracy_s009_full", None)

        save_trial_params_to_file(
            filepath=os.path.join(args_dir, f"top_{rank + 1}_trial.txt"),
            params=trial_params,
            rank=rank + 1,
            trial_id=trial_id,
            loss=trial_loss,
            num_params=trial_num_params,
            model_size_mb=trial_model_size,
            sampler=study.sampler.__class__.__name__,
            val_accuracy=trial_val_acc,
            train_accuracy=trial_train_acc,
            test_accuracy=trial_test_acc,
            test_accuracy_full=trial_test_acc_full,
        )

    # —————————————————————————— Clean-Up Non-Top Trials ————————————————————————— #
    all_trial_ids = {t.number for t in study.trials}
    top_trial_ids = {t.number for t in sorted_trials}

    cleanup_paths = [
        (model_dir, "trial_{trial_id}.keras"),
        (fig_dir, "trial_{trial_id}.png"),
        (history_dir, "trial_{trial_id}.csv"),
    ]

    for trial_id in all_trial_ids - top_trial_ids:
        for base_dir, filename_template in cleanup_paths:
            file_path = os.path.join(base_dir, filename_template.format(trial_id=trial_id))
            if os.path.exists(file_path):
                os.remove(file_path)

    shutil.rmtree(backup_dir, ignore_errors=True)
    if not os.listdir(logs_dir):
        os.rmdir(logs_dir)

    # ——————————————————————————————— Analyze Study —————————————————————————————— #
    analyze_study(study, fig_dir=fig_dir, table_dir=os.path.join(study_dir, "analysis"))

    # ————————————————————————————— End The Training ————————————————————————————— #
    failed_trials = sum(
        1
        for t in study.trials
        if t.state not in {optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED}
    )

    clear_output(wait=True)
    print(f"Training completed with {len(study.trials)} trials.")
    print(f"Number of failed trials: {failed_trials}")
except Exception as e:
    print(f"An error occurred: {e}")
    traceback.print_exc()

    error_log_path = os.path.join(logs_dir, "training_error.log")
    with open(error_log_path, "a") as log_file:
        log_file.write(f"An error occurred during training:\n")
        log_file.write(str(e) + "\n\n")
        log_file.write(traceback.format_exc())
finally:
    notify_training_success(
        recipients_file="./json/recipients.json",
        credentials_file="./json/credentials.json",
        subject=f"🎉 {RUN_DIR} Training Complete",
    )

    stop_monitor(_monitor_proc)

2025-05-24 11:37:54.808705: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_133', 32 bytes spill stores, 32 bytes spill loads

2025-05-24 11:37:54.879507: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_145', 24 bytes spill stores, 24 bytes spill loads

2025-05-24 11:37:54.921210: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_145', 16 bytes spill stores, 16 bytes spill loads

2025-05-24 11:37:55.079557: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_133', 240 bytes spill stores, 240 bytes spill loads

2025-05-24 11:37:55.262894: I external/local_xla/xla/stream_ex



# ——————————————————————— Trial 0 Results ——————————————————————— #

Training loss: 3.104039907455
Training accuracy: 0.1832

Validation loss: 3.104039907455
Validation accuracy: 0.2107

Test loss (s009): 3.027154922485
Test accuracy (s009): 0.2193

Test loss (s009 full): 3.030525445938
Test accuracy (s009 full): 0.2188

Number of parameters: 14185388
Model size: 54.11 MB

# ———————————————————————————————————————————————————————————————————————————— #




Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 1, 1,   │          0 │ input_layer_1[1]… │
│                     │ 2)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ keras_tensor_1CLONE │ (None, 20, 200,   │          0 │ -                 │
│ (InputLayer)        │ 10, 1)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 20, 200,   │          0 │ reshape_1[1][0]   │
│                     │ 10, 2)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 20, 200,   │          0 │ keras_tensor_1CL… │
│ (Concatenate)       │ 10, 3)            │            │ lambda[1][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_0 (Conv3D)    │ (None, 20, 200,   │      2,496 │ concatenate[1][0] │
│                     │ 10, 192)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_bn_0          │ (None, 20, 200,   │        768 │ cnn3d_0[1][0]     │
│ (BatchNormalizatio… │ 10, 192)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_pool_0        │ (None, 20, 66,    │          0 │ cnn3d_bn_0[1][0]  │
│ (MaxPooling3D)      │ 10, 192)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_1 (Conv3D)    │ (None, 20, 66,    │    221,312 │ cnn3d_pool_0[1][… │
│                     │ 10, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_bn_1          │ (None, 20, 66,    │        512 │ cnn3d_1[1][0]     │
│ (BatchNormalizatio… │ 10, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_pool_1        │ (None, 20, 16,    │          0 │ cnn3d_bn_1[1][0]  │
│ (MaxPooling3D)      │ 10, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_2 (Conv3D)    │ (None, 20, 16,    │  2,949,504 │ cnn3d_pool_1[1][… │
│                     │ 10, 384)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_bn_2          │ (None, 20, 16,    │      1,536 │ cnn3d_2[1][0]     │
│ (BatchNormalizatio… │ 10, 384)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_pool_2        │ (None, 20, 3, 10, │          0 │ cnn3d_bn_2[1][0]  │
│ (MaxPooling3D)      │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 230400)    │          0 │ cnn3d_pool_2[1][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ extra_dense_0_laye… │ (None, 100)       │ 23,040,100 │ flatten[1][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ extra_dense_0_drop… │ (None, 100)       │          0 │ extra_dense_0_la… │
│ (Dropout)           │                   │            │                 

 Total params: 26,234,334 (100.08 MB)

 Trainable params: 26,232,926 (100.07 MB)

 Non-trainable params: 1,408 (5.50 KB)

Epoch 1/3


2025-05-24 11:38:10.293265: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:359] gpu_async_0 cuMemAllocAsync failed to allocate 1982857216 bytes: RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
 Reported by CUDA: Free memory/Total memory: 1120075776/4084137984
2025-05-24 11:38:10.293296: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:364] Stats: Limit:                      2388852736
InUse:                      2765407728
MaxInUse:                   2765407728
NumAllocs:                        7778
MaxAllocSize:               2123465008
Reserved:                            0
PeakReserved:                        0
LargestFreeBlock:                    0

2025-05-24 11:38:10.293310: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:68] Histogram of current allocation: (allocation_size_in_bytes, nb_allocation_of_that_sizes), ...;
2025-05-24 11:38:10.293313: E external/local_xla/xla/stream